# Using commit vith onnx quantized

In [ ]:
from utils_dino_final_pylib import *
import onnxruntime
import time

def get_onnx_mask_from_bboxes_onnx(bboxes, ort_session):

    all_masks = []

    for bbox in bboxes:

        onnx_box_coords = bbox.reshape(2, 2)
        onnx_box_labels = np.array([2,3])

        onnx_coord = np.concatenate([ onnx_box_coords], axis=0)[None, :, :]
        onnx_label = np.concatenate([ onnx_box_labels], axis=0)[None, :].astype(np.float32)

        print("onnx_coord before", onnx_coord)


        onnx_coord = predictor.transform.apply_coords(onnx_coord, image.shape[:2]).astype(np.float32)
        
        print("onnx_coord after", onnx_coord)
        print("onnx_label", onnx_label)
        # print("onnx_label.shape", onnx_label.shape)

        print(image.shape[:2])

        onnx_mask_input = np.zeros((1, 1, 256, 256), dtype=np.float32)
        onnx_has_mask_input = np.zeros(1, dtype=np.float32)


        ort_inputs = {
            "image_embeddings": image_embedding,
            "point_coords": onnx_coord,
            "point_labels": onnx_label,
            "mask_input": onnx_mask_input,
            "has_mask_input": onnx_has_mask_input,
            "orig_im_size": np.array(image.shape[:2], dtype=np.float32)
        }

        masks, _, _ = ort_session.run(None, ort_inputs)
        masks = masks > predictor.model.mask_threshold
        print("masks.shape", masks.shape)

        all_masks.append(masks)

    return all_masks

In [ ]:
start_time = time.time()

FOLDER_PATH = r"D:\3d-recon\datasets\ASARoomImage"
# FOLDER_PATH = r"D:\3d-recon\datasets\ASARoomImage\test"

OUTPUT_FOLDER_PATH = r"D:\3d-recon\Grounded-Segment-Anything\v1_final_output_onnx_vith_quantized_rugpoint"

for IMAGE_NAME in os.listdir(FOLDER_PATH):

  SOURCE_IMAGE_PATH = os.path.join(FOLDER_PATH, IMAGE_NAME)

  wall_bboxes, wall_selected_points = get_wall_bboxes_points(SOURCE_IMAGE_PATH)

  print("wall_bboxes", wall_bboxes)
  print("wall_selected_points", wall_selected_points)

  (floor_bboxes, 
  floor_selected_points, 
  rug_points) = get_floor_bboxes_points_with_rug(SOURCE_IMAGE_PATH)

  print("floor_bboxes", floor_bboxes)
  print("floor_selected_points", floor_selected_points)

  # print("rug_bboxes", rug_bboxes)
  # print("rug_to_floor_indices", rug_to_floor_indices)


  output = {'wall_bboxes': wall_bboxes,
        'wall_selected_points':wall_selected_points,
        'floor_bboxes':floor_bboxes,
        'floor_selected_points': floor_selected_points,
        'rug_points': rug_points}
        # 'rug_to_floor_indices':rug_to_floor_indices}

  print("DONE GETTING BBOXES", time.time()-start_time)



  image = cv2.imread(SOURCE_IMAGE_PATH)
  image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(image.shape)

  checkpoint = r"./sam_vit_h_4b8939.pth"

  model_type = "vit_h"
  sam = sam_model_registry[model_type](checkpoint=checkpoint)


  sam.to(device='cuda')
  predictor = SamPredictor(sam)
  predictor.set_image(image)
  image_embedding = predictor.get_image_embedding().cpu().numpy()

  np.save("image_embedding_vith.npy", image_embedding)

  print("DONE SAM EMBEDDING", time.time()-start_time)


  onnx_model_path = r"D:\3d-recon\Grounded-Segment-Anything\sam_quantized.onnx"
  ort_session = onnxruntime.InferenceSession(onnx_model_path)
  wall_masks = get_onnx_mask_from_bboxes_onnx(wall_bboxes, ort_session)

  ort_session = onnxruntime.InferenceSession(onnx_model_path)
  floor_masks = get_onnx_mask_from_bboxes_onnx(floor_bboxes, ort_session)

  # rug_masks = []
  # if len(rug_bboxes)>0:
  #   ort_session = onnxruntime.InferenceSession(onnx_model_path)
  #   rug_masks = get_onnx_mask_from_bboxes_onnx(rug_bboxes, ort_session)

  # Step 1: Convert to uint8 (0 and 255)
  mask_uint8 = (wall_masks[0].squeeze() * 255).astype(np.uint8)  # shape becomes (2657, 1920)


  plt.figure(figsize=(20,20))
  plt.imshow(image)
  for idx, (mask, bbox, point) in enumerate(zip(wall_masks, wall_bboxes, np.array(wall_selected_points))):
      
      print(mask.shape)
      show_mask(mask, plt.gca())
      show_box(bbox, plt.gca())
      # show_points(np.array([point]), labels=np.array([1]), ax=plt.gca())
      # plt.axis('off')
      # plt.savefig(os.path.join(OUTPUT_FOLDER_PATH,f"wall_{idx}_{IMAGE_NAME}") , bbox_inches='tight', pad_inches=0)
      # plt.show()
  

  # plt.imshow(image)
  for idx, (mask, bbox, point) in enumerate(zip(floor_masks, floor_bboxes, np.array(floor_selected_points))):
      # plt.figure(figsize=(20,20))
      # plt.imshow(image)
      show_mask(mask, plt.gca())
      show_box(bbox, plt.gca())
      # show_points(np.array([point]), labels=np.array([1]), ax=plt.gca())
      # plt.axis('off')
      # plt.savefig(os.path.join(OUTPUT_FOLDER_PATH,f"floor_{idx}_{IMAGE_NAME}"), bbox_inches='tight', pad_inches=0)
      # plt.show()
  

  if len(rug_points)>0:
    # plt.imshow(image)

    for point in rug_points:
      print("point", point)
      show_point(point, ax=plt.gca())

  plt.axis('off')
  plt.savefig(os.path.join(OUTPUT_FOLDER_PATH,f"{IMAGE_NAME}.jpg"))
  plt.show()

  # break
